# Практическое занятие № 14 — TimeSeriesSplit, Pipeline, GridSearchCV
**ТюмГУ · Технологии искусственного интеллекта — 1 · 90 минут**

**Цель:** подобрать гиперпараметры регрессионной модели при хронологическом порядке данных, избежать утечек в cross-validation, сравнить CV с holdout и **один раз** проверить финальную модель на test.

**Важно:** `late` и `actual_minutes` нельзя использовать как входы; `actual_minutes` здесь — целевая переменная. `test` не участвует в обучении, подборе, нормировке или выборе между гипотезами.

## 1. Аудит данных и повторение пайплайна (0–17 мин)
Повторяем ту же политику подготовки и то же хронологическое разбиение, что на занятии 13. Проверяем непересечение ID. Изменение правил после изучения теста запрещено.

In [ ]:
# ОБЩАЯ ПОДГОТОВКА: выполняется самостоятельно в каждом блокноте.
# В Colab загрузите orders_dirty.csv через боковую панель «Файлы».
# При отсутствии официального файла используйте только demo_orders_13_14.csv
# и ОТМЕТЬТЕ В ОТЧЁТЕ, что результаты демонстрационные.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('orders_dirty.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('demo_orders_13_14.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv (учебный источник) либо demo_orders_13_14.csv (только демонстрация).')
source_type = 'ОФИЦИАЛЬНЫЙ/УЧЕБНЫЙ НАБОР' if DATA_PATH.name=='orders_dirty.csv' else 'ДЕМОНСТРАЦИОННЫЕ ДАННЫЕ — НЕ СДАВАТЬ КАК ОСНОВНОЙ ИСТОЧНИК'
print('Источник:', DATA_PATH.name, '|', source_type)
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
REQUIRED = {'order_id','created_at','distance_km','queue_size','courier_load','promised_minutes','zone','weather','actual_minutes','late'}
assert REQUIRED.issubset(raw.columns), f'Отсутствуют поля: {sorted(REQUIRED-set(raw.columns))}'
print('Исходный размер:', raw.shape, 'Полных повторов:', int(raw.duplicated().sum()))

# Для регрессии actual_minutes — цель; late запрещено использовать как признак.
# Пустой/нечисловой target не подходит для fit и оценки; диапазон 5–240 —
# учебное правило качества, согласованное со словарём данных, а не физический закон.
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'], errors='coerce')
df['actual_minutes']=pd.to_numeric(df['actual_minutes'], errors='coerce')
valid_target=df['actual_minutes'].between(5,240,inclusive='both')
valid_time=df['created_at'].notna()
valid_id=df['order_id'].notna()
print('Цель вне 5–240 или NaN:',int((~valid_target).sum()),
      'невалидных дат:',int((~valid_time).sum()),'пустых ID:',int((~valid_id).sum()))
df=df.loc[valid_target & valid_time & valid_id].copy()
# Если разные строки имеют один ID, студент обязан разобраться с ним до разбиения.
assert df['order_id'].is_unique, 'Конфликтующие order_id: разберите вручную до продолжения.'
df=df.sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
assert len(df)>=45, f'Для этой демонстрации нужно хотя бы 45 валидных заказов, найдено {len(df)}'
n=len(df)
n_train=int(n*.6); n_dev=int(n*.8)
train=df.iloc[:n_train].copy(); val=df.iloc[n_train:n_dev].copy(); test=df.iloc[n_dev:].copy()
assert len(train)>0 and len(val)>0 and len(test)>0
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
assert train['created_at'].max()<=val['created_at'].min()<=test['created_at'].min()
print('После очистки:',n,'train/val/test:',list(map(len,[train,val,test])))
print('Временные диапазоны:')
for name,part in [('train',train),('val',val),('test (не открывать до занятия 14!)',test)]:
    print(name, len(part),str(part.created_at.min()),'…',str(part.created_at.max()))

NUM_BASE=['distance_km','queue_size','courier_load','promised_minutes']
CAT=['zone','weather']
DERIVED=['hour_sin','hour_cos','planned_speed_kmh']
NUM=NUM_BASE+DERIVED
INPUT=NUM_BASE+CAT+['created_at']

def make_features(X):
    # Функция НЕ обучается по данным и применяется идентично внутри каждого fold.
    f=X.loc[:,INPUT].copy()
    for k in NUM_BASE:
        f[k]=pd.to_numeric(f[k], errors='coerce')
        f.loc[f[k]<0,k]=np.nan
    f.loc[f['promised_minutes']<=0,'promised_minutes']=np.nan
    for k in CAT:
        s=f[k].astype('string').str.strip()
        s=s.mask(s.str.lower().isin(['','nan','none','null','нет данных','неизвестно']))
        f[k]=s.astype(object).where(s.notna(),np.nan)
    ts=pd.to_datetime(f['created_at'],errors='coerce')
    hour=ts.dt.hour.astype(float)
    f['hour_sin']=np.sin(2*np.pi*hour/24)
    f['hour_cos']=np.cos(2*np.pi*hour/24)
    f['planned_speed_kmh']=60*f['distance_km']/f['promised_minutes']
    f['planned_speed_kmh']=f['planned_speed_kmh'].replace([np.inf,-np.inf],np.nan)
    return f.drop(columns=['created_at'])

num_pipe=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='неизвестно')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
prep=ColumnTransformer([('num',num_pipe,NUM),('cat',cat_pipe,CAT)],remainder='drop')

def build_pipeline(regressor):
    return Pipeline([('derive',FunctionTransformer(make_features,validate=False)),
                     ('prep',clone(prep)),('model',regressor)])

def eval_regression(y_true,y_pred):
    return {'MAE':float(mean_absolute_error(y_true,y_pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y_true,y_pred))),
            'R2':float(r2_score(y_true,y_pred))}
Xtrain=train[INPUT].copy(); ytrain=train['actual_minutes'].copy()
Xval=val[INPUT].copy(); yval=val['actual_minutes'].copy()
# test intentionally saved but not used before final check in 14.
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
print('Входные поля:',INPUT,'Цель: actual_minutes (мин)')

## 2. Как устроен TimeSeriesSplit (17–33 мин)

Каждая проверочная часть расположена **позже** обучающей. Размер обучающей части увеличивается по мере перехода от фолда к фолду. Это не означает автоматической защиты от всех временных утечек: признаки должны быть доступны на момент предсказания, а связанные наблюдения не должны пересекать границу train/val.

Для демонстрации используйте `n_splits=4`, `gap=0` (нет дополнительных задержек обновления данных); обоснуйте, когда нужен `gap>0`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Разработка: только первые 80% времени (train+val). Test=последние 20%.
dev=pd.concat([train,val],axis=0).sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
Xdev=dev[INPUT].copy(); ydev=dev['actual_minutes'].copy()
cv=TimeSeriesSplit(n_splits=4)
fold_table=[]
for i,(tr_idx,v_idx) in enumerate(cv.split(Xdev),1):
    fold_table.append({'fold':i,'train_n':len(tr_idx),'val_n':len(v_idx),
        'train_max':str(dev.iloc[tr_idx].created_at.max()),
        'val_min':str(dev.iloc[v_idx].created_at.min()),
        'chronology_ok':bool(dev.iloc[tr_idx].created_at.max()<=dev.iloc[v_idx].created_at.min())})
folds=pd.DataFrame(fold_table)
display(folds)
assert folds['chronology_ok'].all()
folds.to_csv('14_timeseries_folds.csv',index=False,encoding='utf-8-sig')

## 3. Систематический подбор гиперпараметров (33–63 мин)

Ищем конфигурацию с минимальной средней MAE по временным фолдам. В scikit-learn отрицательный `neg_mean_absolute_error` используется потому, что GridSearchCV **максимизирует** оценку. Пишем результат обратно в минутах: `CV_MAE = −mean_test_score`.

На каждом фолде `Pipeline` заново обучает SimpleImputer, StandardScaler, OneHotEncoder и регрессор **только на доступной тренировочной части**.

In [ ]:
searches={
 'Ridge':(
   build_pipeline(Ridge()), {'model__alpha':[0.1,1.0,10.0,100.0]}
 ),
 'RandomForest':(
   build_pipeline(RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1)),
   {'model__max_depth':[4,8,None], 'model__min_samples_leaf':[2,5]}
 )
}
cv_results=[]; grid_models={}
for name,(pipe,grid) in searches.items():
    gs=GridSearchCV(estimator=pipe,param_grid=grid,scoring='neg_mean_absolute_error',
                    cv=cv,refit=True,n_jobs=1,return_train_score=True,error_score='raise')
    gs.fit(Xdev,ydev)
    grid_models[name]=gs
    cv_results.append({'model':name,'CV_MAE':-float(gs.best_score_),
                       'best_params':str(gs.best_params_)})
    print(name,'MAE CV:',round(-gs.best_score_,3),'параметры:',gs.best_params_)
results_cv=pd.DataFrame(cv_results).sort_values('CV_MAE').reset_index(drop=True)
display(results_cv)
results_cv.to_csv('14_cv_summary.csv',index=False,encoding='utf-8-sig')
for name,gs in grid_models.items():
    pd.DataFrame(gs.cv_results_).to_csv(f'14_cv_grid_{name}.csv',index=False,encoding='utf-8-sig')

## 4. Проверка устойчивости и визуализация (63–77 мин)

Посмотрите, в каких фолдах ошибки существенно различаются. Если качество меняется от периода к периоду, один средний показатель скрывает нестабильность. Зафиксируйте выбранный алгоритм и его параметры **до тестирования**.

In [ ]:
winner=str(results_cv.iloc[0]['model'])
best_grid=grid_models[winner]
row=best_grid.best_index_
fold_mae=[-float(best_grid.cv_results_[f'split{i}_test_score'][row]) for i in range(cv.get_n_splits())]
fig,ax=plt.subplots(figsize=(7.3,3.9))
ax.bar(np.arange(1,len(fold_mae)+1),fold_mae)
ax.axhline(np.mean(fold_mae),linestyle='--',label='Средняя MAE')
ax.set(xticks=list(range(1,len(fold_mae)+1)),xlabel='Временной фолд',ylabel='MAE, мин',
       title=f'Устойчивость выбранной модели: {winner}')
ax.legend();fig.tight_layout();fig.savefig('14_mae_by_fold.png',dpi=160);plt.show()

from pathlib import Path
import json
config={'selected_model':winner,'selected_hyperparameters':best_grid.best_params_,
       'selection_metric':'TimeSeriesSplit(n_splits=4), mean MAE (minutes)',
       'cv_mae_minutes':float(-best_grid.best_score_),
       'train_dev_n':int(len(dev)),'test_n':int(len(test)),
       'data_source':DATA_PATH.name,'random_state':42}
Path('14_selected_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Решение зафиксировано до использования test:',config)

## 5. Один финальный тест и ограничения (77–90 мин)

ВНИМАНИЕ: финальный тест проводим **только после записи** `14_selected_config.json`. В этом учебном проекте тестовые заказы могли уже анализироваться в других практиках; поэтому статистику нельзя безоговорочно считать независимой внешней валидацией. Для производственной системы нужен новый будущий период.

После теста не изменяйте гиперпараметры по его результату.

In [ ]:
# Единственная ячейка, обращающаяся к test для оценки модели.
assert Path('14_selected_config.json').exists(), 'Сначала зафиксируйте конфигурацию.'
Xtest=test[INPUT].copy(); ytest=test['actual_minutes'].copy()
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtest.columns)
final_model=best_grid.best_estimator_  # refit=True уже обучил на dev
pred_test=final_model.predict(Xtest)
metrics_test=eval_regression(ytest,pred_test)
print('Зафиксированная модель:',winner)
print('Тестовые метрики:',{k:round(v,3) for k,v in metrics_test.items()})
pd.DataFrame([{'selected_model':winner,'dev_n':len(dev),'test_n':len(test),
              **metrics_test}]).to_csv('14_final_test_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame({'order_id':test['order_id'].values,'actual_minutes':ytest.values,
              'predicted_minutes':pred_test,
              'residual_minutes':ytest.values-pred_test}).to_csv('14_final_test_predictions.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(7.3,3.8))
ax.scatter(pred_test,ytest.values-pred_test,alpha=.75,s=25)
ax.axhline(0,ls='--');ax.set(xlabel='Прогноз, мин',ylabel='Факт − прогноз, мин',title='Остатки на единственном финальном тесте')
fig.tight_layout();fig.savefig('14_final_residuals.png',dpi=160);plt.show()

## 6. Самостоятельная работа и сдача

1. Сравните два лучших набора гиперпараметров; объясните свой выбор по CV_MAE и вариации между фолдами.
2. Для одного фолда укажите максимальную дату обучения и минимальную дату проверки; обоснуйте необходимость `gap`, если данные поступают с задержкой.
3. Объясните, почему `SimpleImputer.fit` до разбиения на фолды создаёт утечку.
4. Приведите **один** финальный тест и отличия от CV. Если ранее test использовался для корректировки модели, явно отметьте это ограничение.
5. Укажите, как вы организовали бы честную проверку на новом месяце заказов.

**Сдать:** `Группа_Фамилия_З14_Отчёт.docx` (или PDF), `14_student.ipynb`, `14_selected_config.json`, CSV с CV и тестом, два PNG. Объём 3–5 страниц без приложений. В Colab запустите «Среда выполнения → Выполнить все» после сброса состояния.